In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW mpsii_diagnosis_and_ERT_table AS
WITH
-- -----------------------------
-- Eligibility (Specified + Incremental Unspecified)
-- -----------------------------
MPSII_Diagnoses_Specified AS (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-07-31'
    UNION ALL
    SELECT DISTINCT PATIENT_ID, FILL_DATE, PRESCRIBER_NPI as NPI
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31'
),
Patients_2Dx_Specified AS (
    SELECT *
    FROM MPSII_Diagnoses_Specified
    where patient_id in (select distinct a.patient_id from MPSII_Diagnoses_Specified as a group by a.patient_id having count(distinct a.fill_date) >= 2)
),
MPSII_Diagnoses_Unspecified AS (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E763%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-07-31'
    UNION ALL
    SELECT DISTINCT PATIENT_ID, FILL_DATE, PRESCRIBER_NPI as NPI
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E763'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31'
),
Patients_2Dx_Unspecified AS (
    SELECT *
    FROM MPSII_Diagnoses_Unspecified
    where patient_id in (select distinct a.patient_id from MPSII_Diagnoses_Unspecified as a group by a.patient_id having count(distinct a.fill_date) >= 2)
),
MPSII_Treatment_All AS (
    SELECT * FROM (
        SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND '2025-07-31'
        UNION ALL
        SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI as NPI
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND TRANSACTION_RESULT = 'PAID'
          AND FILL_DATE BETWEEN '2023-08-01' AND '2025-07-31'
        UNION ALL
        SELECT DISTINCT PATIENT_ID, RENDERING_NPI AS NPI
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                                 '38206','38230','38232','38240','38241','38242','38243','38250')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND '2025-07-31'
    ) t
),
MPSII_Treatment_Elaprase_Only AS (
    SELECT * FROM (
        SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND '2025-07-31'
        UNION ALL
        SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI as NPI
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND TRANSACTION_RESULT = 'PAID'
          AND FILL_DATE BETWEEN '2023-08-01' AND '2025-07-31'
        UNION ALL
        SELECT DISTINCT PATIENT_ID, RENDERING_NPI AS NPI
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE PROCEDURE_CODE = 'J1743'
          AND SERVICE_DATE BETWEEN '2023-08-01' AND '2025-07-31'
    ) t
),
Patients_2Dx_Specified_With_Treatment AS (
    SELECT *
    from MPSII_Treatment_All t  where patient_id in (select distinct a.patient_id from Patients_2Dx_Specified as a)
    --FROM Patients_2Dx_Specified p 
    --INNER JOIN MPSII_Treatment_All t USING (PATIENT_ID)
),
Patients_Incremental_Unspecified AS (
    SELECT DISTINCT p.PATIENT_ID
    FROM Patients_2Dx_Unspecified p
    INNER JOIN MPSII_Treatment_Elaprase_Only t USING (PATIENT_ID)
    WHERE p.PATIENT_ID NOT IN (SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment)
),
eligible_patients AS (
    SELECT * FROM Patients_2Dx_Specified_With_Treatment
    UNION
    --SELECT * FROM Patients_Incremental_Unspecified
    select PATIENT_ID, NPI from MPSII_Diagnoses_Unspecified where patient_id in (select distinct a.patient_id from Patients_Incremental_Unspecified as a) 
),
all_dx_claims_5yr AS (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE, 'Dx' as claim_type
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-07-31'
      AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE, 'Dx' as claim_type
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE IN ('E761','E763')
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31'
      AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
),
--SELECT * from eligible_patients
all_tx_claims_5yr AS (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE, 'Tx' as claim_type
    FROM com_edp_prd.com_raw.kom_medical_events 
    WHERE NDC11 IN ('54092070001','540920700')
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-07-31'
      AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE, 'Tx' as claim_type
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE NDC11 IN ('54092070001','540920700')
      AND TRANSACTION_RESULT = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31'
      AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    UNION
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE, 'Tx' as claim_type
    FROM com_edp_prd.com_raw.kom_medical_events 
    WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                             '38206','38230','38232','38240','38241','38242','38243','38250')
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-07-31'
      AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
),
all_claims_5yr AS (
    SELECT * FROM all_dx_claims_5yr
    UNION
    SELECT * FROM all_tx_claims_5yr
),
-- SELECT * FROM all_claims_5yr
-- WHERE patient_id = 'XBV2CVLN' and NPI  = '1629080833'
diagnosis_counts AS (
    -- Count distinct diagnosis dates per patient
    SELECT 
        PATIENT_ID,
        COUNT(DISTINCT fill_date) AS dx_claim_count
    FROM all_dx_claims_5yr
    GROUP BY PATIENT_ID
),
treatment_counts AS (
    -- Count distinct treatment dates per patient
    SELECT 
        PATIENT_ID,
        COUNT(DISTINCT fill_date) AS tx_claim_count
    FROM all_tx_claims_5yr
    GROUP BY PATIENT_ID
),
patient_demographics AS (
    -- Get patient demographics with standardized gender values
    SELECT 
        PATIENT_ID,
        AGE,
        CASE 
            WHEN AGE <= 5 THEN '≤5'
            ELSE '>5'
        END AS AGE_GROUP,
        CASE 
            WHEN UPPER(PATIENT_GENDER) IN ('M', 'MALE') THEN 'M'
            WHEN UPPER(PATIENT_GENDER) IN ('F', 'FEMALE') THEN 'F'
            ELSE PATIENT_GENDER
        END AS GENDER
    FROM (
        SELECT 
            PATIENT_ID,
            PATIENT_GENDER,
            (2025 - YEAR(PATIENT_YOB)) AS AGE
        FROM com_edp_prd.com_raw.kom_patient_demographics
        WHERE patient_id in (SELECT distinct patient_id from eligible_patients)
    )
),
--SELECT count(distinct patient_id) FROM patient_demographics
-- --WHERE patient_id = '6XZ63TZJ'

patient_clinical_data AS (
    -- Combine diagnosis and treatment counts
    SELECT 
        COALESCE(d.PATIENT_ID, t.PATIENT_ID) AS PATIENT_ID,
        COALESCE(d.dx_claim_count, 0) AS dx_claim_count,
        COALESCE(t.tx_claim_count, 0) AS tx_claim_count
    FROM diagnosis_counts d
    FULL OUTER JOIN treatment_counts t
        ON d.PATIENT_ID = t.PATIENT_ID
),
-- Final cohort assignment
patient_base as (SELECT 
    dem.PATIENT_ID,
    dem.AGE,
    dem.AGE_GROUP,
    dem.GENDER,
    CASE 
        -- Younger Male: Male, Age ≤5, (2+ Dx OR 1+ Dx + 2+ Tx)
        WHEN dem.GENDER = 'M' 
            AND dem.AGE <= 5 
            AND (clin.dx_claim_count >= 2 OR (clin.dx_claim_count >= 1 AND clin.tx_claim_count >= 2))
        THEN 'Younger Male'
        
        -- Older Male: Male, Age >5, (20+ Dx OR 1+ Dx + 2+ Tx)
        WHEN dem.GENDER = 'M' 
            AND dem.AGE > 5 
            AND (clin.dx_claim_count >= 20 OR (clin.dx_claim_count >= 1 AND clin.tx_claim_count >= 2))
        THEN 'Older Male'
        
        -- Female: Female, Any Age, (20+ Dx OR 1+ Dx + 2+ Tx)
        WHEN dem.GENDER = 'F' 
            AND (clin.dx_claim_count >= 20 OR (clin.dx_claim_count >= 1 AND clin.tx_claim_count >= 2))
        THEN 'Female'
        
        ELSE NULL
    END AS COHORT
FROM patient_demographics dem
INNER JOIN patient_clinical_data clin
    ON dem.PATIENT_ID = clin.PATIENT_ID
WHERE (clin.dx_claim_count >= 1  -- Must have at least 1 E76.1 diagnosis
  AND CASE 
        WHEN dem.GENDER = 'M' AND dem.AGE <= 5 
            AND (clin.dx_claim_count >= 2 OR (clin.dx_claim_count >= 1 AND clin.tx_claim_count >= 2))
        THEN 1
        WHEN dem.GENDER = 'M' AND dem.AGE > 5 
            AND (clin.dx_claim_count >= 20 OR (clin.dx_claim_count >= 1 AND clin.tx_claim_count >= 2))
        THEN 1
        WHEN dem.GENDER = 'F' 
            AND (clin.dx_claim_count >= 20 OR (clin.dx_claim_count >= 1 AND clin.tx_claim_count >= 2))
        THEN 1
        ELSE 0
      END = 1)  -- Only include patients who qualify for a cohort
ORDER BY COHORT, AGE, PATIENT_ID),

base AS (
    SELECT DISTINCT
        a.patient_id,
        b.service_date,
        b.diagnosis_codes
    FROM patient_base a
    LEFT JOIN com_edp_prd.com_raw.kom_medical_events b
        ON a.patient_id = b.patient_id
       AND b.service_date BETWEEN '2020-08-01' AND '2025-07-31'
),
flagged AS (
    SELECT
        patient_id,
        service_date,
        CASE
            WHEN diagnosis_codes IS NOT NULL
             AND (
                diagnosis_codes ILIKE '%|G910|%' OR diagnosis_codes ILIKE '%|G911|%' OR
                diagnosis_codes ILIKE '%|G912|%' OR diagnosis_codes ILIKE '%|G913|%' OR
                diagnosis_codes ILIKE '%|G914|%' OR diagnosis_codes ILIKE '%|G918|%' OR
                diagnosis_codes ILIKE '%|G919|%' OR diagnosis_codes ILIKE '%|Q038|%' OR
                diagnosis_codes ILIKE '%|Q039|%' OR diagnosis_codes ILIKE '%|Q050|%' OR
                diagnosis_codes ILIKE '%|Q051|%' OR diagnosis_codes ILIKE '%|Q052|%' OR
                diagnosis_codes ILIKE '%|Q053|%' OR diagnosis_codes ILIKE '%|Q054|%' OR
                diagnosis_codes ILIKE '%|Q055|%' OR diagnosis_codes ILIKE '%|Q056|%' OR
                diagnosis_codes ILIKE '%|Q057|%' OR diagnosis_codes ILIKE '%|Q058|%' OR
                diagnosis_codes ILIKE '%|Q0700|%' OR diagnosis_codes ILIKE '%|Q0702|%' OR
                diagnosis_codes ILIKE '%|Q0703|%' OR diagnosis_codes ILIKE '%|F445|%' OR
                diagnosis_codes ILIKE '%|F639|%' OR diagnosis_codes ILIKE '%|F70|%' OR
                diagnosis_codes ILIKE '%|F71|%' OR diagnosis_codes ILIKE '%|F72|%' OR
                diagnosis_codes ILIKE '%|F73|%' OR diagnosis_codes ILIKE '%|F78|%' OR
                diagnosis_codes ILIKE '%|F78A1|%' OR diagnosis_codes ILIKE '%|F78A9|%' OR
                diagnosis_codes ILIKE '%|F79|%' OR diagnosis_codes ILIKE '%|F800|%' OR
                diagnosis_codes ILIKE '%|F801|%' OR diagnosis_codes ILIKE '%|F802|%' OR
                diagnosis_codes ILIKE '%|F804|%' OR diagnosis_codes ILIKE '%|F8081|%' OR
                diagnosis_codes ILIKE '%|F8082|%' OR diagnosis_codes ILIKE '%|F8089|%' OR
                diagnosis_codes ILIKE '%|F809|%' OR diagnosis_codes ILIKE '%|F810|%' OR
                diagnosis_codes ILIKE '%|F812|%' OR diagnosis_codes ILIKE '%|F8181|%' OR
                diagnosis_codes ILIKE '%|F8189|%' OR diagnosis_codes ILIKE '%|F819|%' OR
                diagnosis_codes ILIKE '%|F82|%' OR diagnosis_codes ILIKE '%|F840|%' OR
                diagnosis_codes ILIKE '%|F843|%' OR diagnosis_codes ILIKE '%|F845|%' OR
                diagnosis_codes ILIKE '%|F848|%' OR diagnosis_codes ILIKE '%|F849|%' OR
                diagnosis_codes ILIKE '%|F88|%' OR diagnosis_codes ILIKE '%|F89|%' OR
                diagnosis_codes ILIKE '%|R6250|%' OR diagnosis_codes ILIKE '%|R620|%' OR
                diagnosis_codes ILIKE '%|R6251|%' OR diagnosis_codes ILIKE '%|R6259|%' OR
                diagnosis_codes ILIKE '%|R62|%'
             )
            THEN 1 ELSE 0
        END AS has_black_code
    FROM base
),
severity_by_patient AS (
    SELECT
        patient_id,
        COUNT(DISTINCT service_date) AS count_fill_date,
        COUNT(DISTINCT CASE WHEN has_black_code = 1 THEN service_date END) AS black_dx_distinct_dates,
        CASE
            WHEN COUNT(DISTINCT CASE WHEN has_black_code = 1 THEN service_date END) >= 2
            THEN 'Severe'
            ELSE 'Attenuated'
        END AS severity
    FROM flagged
    GROUP BY patient_id
)
SELECT b.*, a.severity FROM severity_by_patient a
INNER JOIN patient_base b
ON b.patient_id = a.patient_id








In [0]:
%sql
SELECT *,
case when age <5  then 'Newly Diagnosed'
when ((age >=5 and age <=10)) then 'Seeking New Treatment'
when age > 10 and severity = 'Severe' then 'Disengaged'
when age > 10 and severity = 'Attenuated' then 'Status Quo'
END as Patient_Categorization
from mpsii_diagnosis_and_ERT_table

In [0]:
case when age <5 then 'Newly Diagnosed'
when ((age >=5 and age <=10)) or has_intrathecal_device = 1 then 'Seeking New Treatment'
when age > 10 and severity = 'Severe' then 'Disengaged'
when age > 10 and severity = 'Attenuated' then 'Status Quo'

In [0]:
%sql
CREATE OR REPLACE temporary VIEW mpsii_diagnosed_patient_categorization AS
WITH intrathecal_base AS (
    SELECT DISTINCT d.*,

case when age <5 then 'Newly Diagnosed'
when ((age >=5 and age <=10)) or has_intrathecal_device = 1 then 'Seeking New Treatment'
when age > 10 and severity = 'Severe' then 'Disengaged'
when age > 10 and severity = 'Attenuated' then 'Status Quo'
End as patient_category
from mpsii_diagnosed_demo_severity_intrathecal d
) 
SELECT * FROM intrathecal_base

Final List : Patient - Patient Category

In [0]:
%sql
SELECT DISTINCT patient_id, patient_category
FROM mpsii_diagnosed_patient_categorization